# Occupation family classification

Census fields cannot identify one of the original 14 occupation titles reliably (about 36% accuracy). This notebook trains six models on two broader label schemes and keeps the run with the highest holdout accuracy.

**How to run in Google Colab**

1. Upload this notebook to Colab and open it.
2. Run the cells from top to bottom.
3. When the data cell asks for a file, upload `adult_clean.csv`.
4. Training takes a few minutes. The best model is saved at the end, and Colab offers a zip download of the results.

**Label schemes**

- Education-aligned families: Professional-managerial, Clerical-sales-service, Manual
- Collar groups: White-collar, Blue-collar, Service

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RESULTS_PATH = Path("occupation_model_results.csv")
MODEL_PATH = Path("best_occupation_model.joblib")
COMPARISON_PATH = Path("model_comparison.png")
CONFUSION_PATH = Path("best_confusion_matrix.png")


## Load the dataset

If `adult_clean.csv` is not already in the Colab session, this cell opens a file picker.

In [ ]:
DATA_PATH = Path("adult_clean.csv")

if not DATA_PATH.exists():
    from google.colab import files

    print("Upload adult_clean.csv")
    uploaded = files.upload()
    if not DATA_PATH.exists():
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_names) != 1:
            raise FileNotFoundError("Upload the file named adult_clean.csv.")
        DATA_PATH = Path(csv_names[0])

raw_df = pd.read_csv(DATA_PATH)
print("Dataset loaded:", DATA_PATH)
print("Shape:", raw_df.shape)
raw_df.head()


## Clean the table and define the two occupation groupings

`source` is only the original train/test flag. `education` repeats `education-num`. `fnlwgt` is a census sampling weight. Armed-Forces has 14 rows, which is too small to learn. Rare countries are grouped as Other, and capital gain and loss are log-transformed.

In [ ]:
DROPPED_OCCUPATIONS = ["Armed-Forces"]
RARE_COUNTRY_MIN = 100

EDUCATION_ALIGNED = {
    "Prof-specialty": "Professional-managerial",
    "Exec-managerial": "Professional-managerial",
    "Tech-support": "Professional-managerial",
    "Adm-clerical": "Clerical-sales-service",
    "Sales": "Clerical-sales-service",
    "Other-service": "Clerical-sales-service",
    "Priv-house-serv": "Clerical-sales-service",
    "Protective-serv": "Clerical-sales-service",
    "Craft-repair": "Manual",
    "Machine-op-inspct": "Manual",
    "Transport-moving": "Manual",
    "Handlers-cleaners": "Manual",
    "Farming-fishing": "Manual",
}

COLLAR_GROUPS = {
    "Prof-specialty": "White-collar",
    "Exec-managerial": "White-collar",
    "Tech-support": "White-collar",
    "Adm-clerical": "White-collar",
    "Sales": "White-collar",
    "Craft-repair": "Blue-collar",
    "Machine-op-inspct": "Blue-collar",
    "Transport-moving": "Blue-collar",
    "Handlers-cleaners": "Blue-collar",
    "Farming-fishing": "Blue-collar",
    "Other-service": "Service",
    "Priv-house-serv": "Service",
    "Protective-serv": "Service",
}

LABEL_SCHEMES = {
    "Education-aligned families": EDUCATION_ALIGNED,
    "Collar groups": COLLAR_GROUPS,
}


def load_and_clean(frame):
    df = frame.copy()
    df = df.drop(columns=["source", "education", "fnlwgt"])
    df = df[~df["occupation"].isin(DROPPED_OCCUPATIONS)].copy()

    df["capital-gain"] = np.log1p(df["capital-gain"])
    df["capital-loss"] = np.log1p(df["capital-loss"])

    country_counts = df["native-country"].value_counts()
    rare_countries = set(country_counts[country_counts < RARE_COUNTRY_MIN].index)
    df["native-country"] = df["native-country"].where(
        ~df["native-country"].isin(rare_countries), "Other"
    )

    df["age-bin"] = pd.cut(
        df["age"], bins=[0, 25, 35, 45, 55, 65, 100], labels=False
    )
    df["hours-bin"] = pd.cut(
        df["hours-per-week"], bins=[0, 20, 35, 40, 50, 99], labels=False
    )
    df["has-capital-gain"] = (df["capital-gain"] > 0).astype(int)
    df["has-capital-loss"] = (df["capital-loss"] > 0).astype(int)
    df["edu-x-hours"] = df["education-num"] * df["hours-per-week"]
    df["is-us"] = (df["native-country"] == "United-States").astype(int)
    return df


df = load_and_clean(raw_df)
print("Cleaned dataset shape:", df.shape)
print("Features:", [column for column in df.columns if column != "occupation"])
df.head()


## Models

Each label scheme is trained with logistic regression, a decision tree, random forest, extra trees, histogram gradient boosting, and a soft vote of boosting, random forest, and logistic regression. The highest holdout accuracy is kept.

In [ ]:
def make_preprocessor(X):
    categorical = X.select_dtypes(include=["object"]).columns.tolist()
    numerical = X.select_dtypes(exclude=["object"]).columns.tolist()
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                categorical,
            ),
            ("numerical", StandardScaler(), numerical),
        ]
    )


def make_hgb():
    return HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.06,
        max_depth=6,
        min_samples_leaf=30,
        random_state=42,
    )


def make_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=16, min_samples_leaf=5, random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42,
        ),
        "Hist Gradient Boosting": make_hgb(),
        "Soft Vote": VotingClassifier(
            estimators=[
                ("hgb", make_hgb()),
                (
                    "rf",
                    RandomForestClassifier(
                        n_estimators=300,
                        min_samples_leaf=2,
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
                ("lr", LogisticRegression(max_iter=2000)),
            ],
            voting="soft",
        ),
    }


def train_all(frame):
    features = frame.drop(columns=["occupation"])
    rows = []
    trained = {}

    for scheme_name, mapping in LABEL_SCHEMES.items():
        missing = set(frame["occupation"].unique()) - set(mapping)
        if missing:
            raise ValueError(f"{scheme_name} is missing occupations: {missing}")

        target = frame["occupation"].map(mapping)
        majority = target.value_counts().iloc[0] / len(target)
        X_train, X_test, y_train, y_test = train_test_split(
            features,
            target,
            test_size=0.20,
            random_state=42,
            stratify=target,
        )

        print("\n" + "=" * 70)
        print(scheme_name)
        print("=" * 70)
        print("Class counts:")
        print(target.value_counts().to_string())
        print(f"Majority baseline: {majority:.4f}")

        for model_name, model in make_models().items():
            pipeline = Pipeline(
                steps=[
                    ("preprocessing", make_preprocessor(features)),
                    ("model", model),
                ]
            )
            pipeline.fit(X_train, y_train)
            predictions = pipeline.predict(X_test)

            accuracy = accuracy_score(y_test, predictions)
            macro_f1 = f1_score(y_test, predictions, average="macro")
            weighted_f1 = f1_score(y_test, predictions, average="weighted")
            print(
                f"{model_name:24}  accuracy {accuracy:.4f}  "
                f"macro F1 {macro_f1:.4f}"
            )

            trained[(scheme_name, model_name)] = {
                "pipeline": pipeline,
                "X_test": X_test,
                "y_test": y_test,
                "predictions": predictions,
            }
            rows.append(
                {
                    "Label scheme": scheme_name,
                    "Model": model_name,
                    "Accuracy": accuracy,
                    "Macro F1": macro_f1,
                    "Weighted F1": weighted_f1,
                    "Majority baseline": majority,
                }
            )

    results = pd.DataFrame(rows)
    results = results.sort_values(
        by=["Accuracy", "Macro F1"], ascending=False
    ).reset_index(drop=True)
    return results, trained


results, trained = train_all(df)
results


## Compare models and keep the most accurate one

In [ ]:
results.to_csv(RESULTS_PATH, index=False)

schemes = list(LABEL_SCHEMES)
model_names = list(make_models())
x = np.arange(len(model_names))
width = 0.36

fig, ax = plt.subplots(figsize=(13, 6))
for index, scheme in enumerate(schemes):
    subset = results[results["Label scheme"] == scheme].set_index("Model")
    scores = [subset.loc[model, "Accuracy"] for model in model_names]
    offset = (index - 0.5) * width
    ax.bar(x + offset, scores, width, label=scheme)

ax.axhline(0.60, color="gray", linestyle="--", linewidth=1, label="60% target")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Holdout accuracy")
ax.set_title("Occupation family models")
ax.legend()
fig.tight_layout()
fig.savefig(COMPARISON_PATH, dpi=150)
plt.show()

best = results.iloc[0]
best_run = trained[(best["Label scheme"], best["Model"])]

print("SELECTED MODEL")
print("Label scheme:", best["Label scheme"])
print("Model:", best["Model"])
print("Accuracy:", round(best["Accuracy"], 4))
print("Macro F1:", round(best["Macro F1"], 4))
print("Weighted F1:", round(best["Weighted F1"], 4))
print("Majority baseline:", round(best["Majority baseline"], 4))
print()
print(classification_report(best_run["y_test"], best_run["predictions"], digits=4))

labels = sorted(best_run["y_test"].unique())
matrix = confusion_matrix(best_run["y_test"], best_run["predictions"], labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(matrix, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix - {best['Model']} ({best['Label scheme']})")
for row in range(matrix.shape[0]):
    for col in range(matrix.shape[1]):
        ax.text(col, row, str(matrix[row, col]), ha="center", va="center")
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(CONFUSION_PATH, dpi=150)
plt.show()

joblib.dump(
    {
        "pipeline": best_run["pipeline"],
        "label_scheme": best["Label scheme"],
        "model_name": best["Model"],
        "accuracy": float(best["Accuracy"]),
        "label_map": LABEL_SCHEMES[best["Label scheme"]],
    },
    MODEL_PATH,
)

examples = pd.DataFrame(
    {
        "Actual": best_run["y_test"].values[:20],
        "Predicted": best_run["predictions"][:20],
    }
)
print("Example predictions:")
examples


## Download the results

On Colab this cell downloads one zip containing the comparison table, both charts, and the saved model.

In [ ]:
import zipfile

zip_path = Path("occupation_results.zip")
with zipfile.ZipFile(zip_path, "w") as archive:
    for path in (RESULTS_PATH, COMPARISON_PATH, CONFUSION_PATH, MODEL_PATH):
        archive.write(path, path.name)

print("Saved:")
for path in (RESULTS_PATH, COMPARISON_PATH, CONFUSION_PATH, MODEL_PATH, zip_path):
    print("-", path.resolve())

try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print("Not running in Colab, so the zip was saved next to the notebook only.")
